# Week 6 - Apache Spark Assignment

**Name:** Mohit Poonia

## Objective

Understand Spark architecture and perform efficient data processing using transformations, filtering, schema handling, optimized file formats, and Spark DataFrame operations.

## Table of Contents

1. Spark Session
2. Dataset Loading
3. Q1 - Spark Architecture
4. Q2 - Lazy Evaluation
5. Q3 - Reading CSV Files
6. Q4 - CSV vs Parquet
7. Q5 - Filtering and Selection
8. Q6 - Renaming Columns and Casting
9. Q7 - DAG and Fault Tolerance
10. Q8 - Filtering with AND
11. Q9 - Predicate Pushdown
12. Q10 - Creating New Columns
13. Q11 - Transformations vs Actions
14. Q12 - Read → Filter → Write Pipeline
15. Q13 - Client vs Cluster Mode
16. Q14 - Filtering with OR
17. Q15 - show() vs collect()
18. Conclusion

## Technologies Used

- Apache Spark (PySpark)
- Python
- Jupyter Notebook
- CSV
- Parquet

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Week6 Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [2]:
df = spark.read.csv(
    r"C:\Users\Mohit\Downloads\week5_dataset.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|         Fashion|        300| Delhi| 19|       Basic|    NULL|300.0|       1|user2@gmail.com|   priya|2025-01-02 11:00:00|
|    103|      2025-01-03|  West|     Electronics|        700|Mu

In [3]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)



In [4]:
df.count()

10

## Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Answer:

- **Driver:** The Driver is the brain of a Spark application. It receives the user program, creates the execution plan (DAG), requests resources from the Cluster Manager, assigns tasks to Executors, and collects the final results.

- **Cluster Manager:** The Cluster Manager is responsible for managing cluster resources. It allocates resources and launches Executors on worker nodes based on the Driver's request.

- **Executor:** Executors are worker processes that execute the tasks assigned by the Driver. They process data, perform computations, and return the results to the Driver.

## Q2. How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

### Answer:

Spark uses Lazy Evaluation by delaying the execution of transformations until an Action is called. During this time, Spark builds a Directed Acyclic Graph (DAG) of all transformations and optimizes the execution plan. This reduces unnecessary computations, disk I/O, and memory usage, making the processing of large datasets faster and more efficient.

## Q3. Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [5]:
df = spark.read.csv(
    r"C:\Users\Mohit\Downloads\week5_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|price|store_id|          email|username|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|
|    102|      2025-01-02|  East|         Fashion|        300| Delhi| 19|       Basic|    NULL|300.0|       1|user2@gmail.com|   priya|2025-01-02 11:00:00|
|    103|      2025-01-03|  West|     Electronics|        700|Mu

### Insight

The `spark.read.csv()` method is used to load CSV files into a Spark DataFrame. Setting `header=True` treats the first row as column names, while `inferSchema=True` automatically detects the appropriate data types for each column, making data analysis easier.

## Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

### Answer

- **CSV** is a **row-based** storage format, meaning data is stored row by row. When Spark reads a CSV file, it often needs to read the entire row even if only a few columns are required.

- **Parquet** is a **columnar** storage format, meaning data is stored column by column. Spark can read only the required columns, reducing disk I/O and memory usage.

- **Performance Difference:** Parquet provides better performance for analytical workloads because it supports column pruning, compression, and Predicate Pushdown, allowing Spark to process large datasets more efficiently than CSV.

## Q5. Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the category is `'Electronics'`.

In [6]:
df.filter(
    df.product_category == "Electronics"
).select(
    "user_id",
    "price"
).show()

+-------+-----+
|user_id|price|
+-------+-----+
|    101|500.0|
|    101|500.0|
|    103| NULL|
|    107|800.0|
+-------+-----+



### Insight

The `filter()` transformation was used to retrieve only the rows where the `product_category` is **Electronics**, and the `select()` transformation displayed only the required columns (`user_id` and `price`). The `show()` action executed the query and displayed the filtered results.

## Q6. Write the code to revise a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.

In [7]:
from pyspark.sql.types import DoubleType

df_updated = df.withColumnRenamed(
    "username",
    "customer_name"
).withColumn(
    "price",
    df.price.cast(DoubleType())
)

df_updated.printSchema()
df_updated.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+-------------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|price|store_id|          email|customer_name|      raw_timestamp|
+-------+----------------+------+----------------+-----------+------+---+------------+--------

### Insight

The `withColumnRenamed()` transformation was used to rename the `username` column to `customer_name`. The `cast()` function converted the `price` column to the `Double` data type. These transformations improve schema consistency and ensure that numeric columns can be used for calculations.

## Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

### Answer

Spark maintains a Lineage Graph (DAG) that records all the transformations applied to the data. If an Executor (worker node) fails, Spark uses this DAG to identify the lost partitions and recomputes only the missing data instead of reprocessing the entire dataset. This provides efficient fault tolerance and minimizes recovery time.

## Q8. Write a query to filter a DataFrame `df_orders` for rows where the `status` is `'Completed'` AND the `amount` is greater than `1000`.

In [8]:
df.filter(
    (df.status == "Completed") &
    (df.sale_amount > 1000)
).show()

+-------+----------------+------+----------------+-----------+----+---+------------+------+-----+--------+-----+--------+-------------+
|user_id|transaction_date|region|product_category|sale_amount|city|age|subscription|status|price|store_id|email|username|raw_timestamp|
+-------+----------------+------+----------------+-----------+----+---+------------+------+-----+--------+-----+--------+-------------+
+-------+----------------+------+----------------+-----------+----+---+------------+------+-----+--------+-----+--------+-------------+



### Insight

The `filter()` transformation was used with the **AND (`&`)** operator to retrieve only the rows where the order status is **Completed** and the `sale_amount` is greater than **1000**. The `show()` action executed the query and displayed the filtered records.

## Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

### Answer

Predicate Pushdown is an optimization technique used by Spark with Parquet files. Instead of loading the entire dataset into memory and then applying filters, Spark pushes the filter condition to the Parquet file itself. As a result, only the required rows are read into memory, reducing disk I/O, memory usage, and improving query performance.

## Q10. Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by `1.18` (18% tax).

In [9]:
df.withColumn(
    "final_price",
    df.price * 1.18
).show()

+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+-----------+
|user_id|transaction_date|region|product_category|sale_amount|  city|age|subscription|  status|price|store_id|          email|username|      raw_timestamp|final_price|
+-------+----------------+------+----------------+-----------+------+---+------------+--------+-----+--------+---------------+--------+-------------------+-----------+
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|      590.0|
|    101|      2025-01-01|  West|     Electronics|        500|Jaipur| 25|     Premium|  Active|500.0|       1|user1@gmail.com|   rahul|2025-01-01 10:00:00|      590.0|
|    102|      2025-01-02|  East|         Fashion|        300| Delhi| 19|       Basic|    NULL|300.0|       1|user2@gmail.com|   priya|2025-01-02 11:00:00|     

### Insight

The `withColumn()` transformation was used to create a new column named `final_price` by multiplying the existing `price` column by **1.18**. This keeps the original data unchanged while creating a calculated column for further analysis.

## Q11. What is the difference between Transformations and Actions? Provide two examples of each.

### Answer

**Transformations** are operations that create a new DataFrame but do not execute immediately. Spark records these operations and executes them only when an Action is called.

**Examples:**
- `filter()`
- `select()`

**Actions** trigger the execution of all pending transformations and produce a result.

**Examples:**
- `show()`
- `count()`

## Q12. Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where `user_id` is null, and save the result as a CSV at "path/to/output".

In [ ]:
df_parquet = spark.read.parquet("path/to/input")

df_parquet.filter(
    df_parquet.user_id.isNotNull()
).write.csv(
    "path/to/output",
    header=True,
    mode="overwrite"
)

**Note:** This code is provided as per the assignment specification. Since the required Parquet input file is not available, the command has not been executed. The logic demonstrates how to read a Parquet file, filter null values, and save the cleaned data as a CSV.

### Insight

The Parquet file is loaded into a DataFrame, rows with null `user_id` values are removed using `filter()` and `isNotNull()`, and the cleaned data is written as a CSV file using `write.csv()`.

## Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

### Answer

**Client Mode:** The Driver runs on the client machine (local computer), while Executors run on the cluster. If the client machine shuts down, the Spark application may stop.

**Cluster Mode:** Both the Driver and Executors run inside the cluster. The application continues running even if the client machine disconnects, making it suitable for production environments.

## Q14. Write a query to filter a dataset for rows where the `region` is `'North'` OR the `priority` is `'High'`.

In [ ]:
df.filter(
    (df.region == "North") |
    (df.subscription == "Premium")
).show()

### Insight

The `filter()` transformation combined with the OR (`|`) operator retrieves rows that satisfy either condition. The `show()` action executes the query and displays the matching records.

## Q15. When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

### Answer

The `show(5)` action displays only a small preview of the dataset, making it memory-efficient and suitable for exploring large datasets. In contrast, `collect()` retrieves the entire dataset to the Driver's memory, which can cause excessive memory usage or even application failure when working with very large datasets.

# Conclusion

In this assignment, Apache Spark concepts such as Spark Architecture, Lazy Evaluation, DAG, Transformations, Actions, DataFrame operations, schema handling, CSV and Parquet file formats, Predicate Pushdown, Client and Cluster execution modes, and Spark data pipelines were implemented and analyzed. The exercises demonstrated how Spark efficiently processes large-scale datasets while optimizing performance and resource utilization.